In [5]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src" / "utils_cv.py").exists():
    root = root.parent

sys.path.insert(0, str(root / "src"))
print(Path.cwd())

c:\Users\Lenovo\Documents\GitHub\SIMUTEH-2330511060\notebook


# Kode 4.1 Verifikasi konvolusi manual 

In [6]:
import time 
import cv2
import numpy as np
from utils_cv import baca, kontras_bercak
  
def konvolusi(img, k):            # fungsi dari slide P4 
    kh, kw = k.shape; ph, pw = kh // 2, kw // 2 
    pad = np.pad(img.astype(float), ((ph, ph), (pw, pw)), mode="edge") 
    out = np.zeros(img.shape) 
    for y in range(img.shape[0]): 
        for x in range(img.shape[1]): 
            out[y, x] = np.sum(pad[y:y+kh, x:x+kw] * k) 
    return np.clip(np.round(out), 0, 255).astype(np.uint8) 
  
def konvolusi_sejati(img, k):     # kernel dibalik 180 derajat dahulu 
    return konvolusi(img, np.flip(k)) 
  
nama = "teh_041.png" 
bersih = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nama}"), 
                      cv2.COLOR_BGR2GRAY) 
md = cv2.imread(f"../data/sintetis/mask_daun/{nama}", 0) 
mb = cv2.imread(f"../data/sintetis/mask_bercak/{nama}", 0) 
  
for nm, k in {"rerata": np.ones((3, 3)) / 9, 
              "asimetris": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float) 
              }.items(): 
    t0 = time.perf_counter(); a = konvolusi(bersih, k) 
    t_manual = time.perf_counter() - t0 
    t0 = time.perf_counter() 
    b = cv2.filter2D(bersih, -1, k, borderType=cv2.BORDER_REPLICATE) 
    t_cv = time.perf_counter() - t0 
    beda = np.abs(a.astype(int) - b.astype(int)).max() 
    c = konvolusi_sejati(bersih, k) 
    beda2 = np.abs(c.astype(int) - b.astype(int)).max()
    print(f"{nm}: selisih manual vs filter2D={beda}, " 
          f"sejati vs filter2D={beda2}, " 
          f"waktu manual={t_manual:.3f} s, filter2D={t_cv:.5f} s")

rerata: selisih manual vs filter2D=0, sejati vs filter2D=0, waktu manual=1.807 s, filter2D=0.00052 s
asimetris: selisih manual vs filter2D=0, sejati vs filter2D=255, waktu manual=1.841 s, filter2D=0.00038 s


# Kode 4.2 Tabel penapis x derau x PSNR x waktu x KB 

In [7]:
import pandas as pd 
rng = np.random.default_rng(42) 
  
def derau_gauss(img, sd): 
    return np.clip(img + rng.normal(0, sd, img.shape), 0, 255).astype(np.uint8) 
  
def derau_sp(img, p):                  # p = proporsi piksel rusak 
    out = img.copy(); m = rng.random(img.shape) 
    out[m < p / 2] = 0; out[m > 1 - p / 2] = 255 
    return out 
  
def ukur_waktu(f, x, ulang=20):        # rerata beberapa kali, dalam ms 
    t = [] 
    for _ in range(ulang): 
        t0 = time.perf_counter(); f(x); t.append(time.perf_counter() - t0) 
    return 1000 * np.mean(t) 
  
derau = {"gauss_10": derau_gauss(bersih, 10), "gauss_20": derau_gauss(bersih, 
20), 
         "gauss_40": derau_gauss(bersih, 40), "sp_4%": derau_sp(bersih, 0.04), 
         "sp_10%": derau_sp(bersih, 0.10)} 
penapis = {"rerata_3": lambda x: cv2.blur(x, (3, 3)), 
           "rerata_5": lambda x: cv2.blur(x, (5, 5)), 
           "gauss_5": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0), 
           "median_3": lambda x: cv2.medianBlur(x, 3), 
           "median_5": lambda x: cv2.medianBlur(x, 5), 
           "median_9": lambda x: cv2.medianBlur(x, 9)} 
baris = [] 
for dn, d in derau.items(): 
    baris.append([dn, "tanpa", cv2.PSNR(bersih, d), 0.0, 
                  kontras_bercak(d, md, mb)]) 
    for pn, f in penapis.items(): 
        out = f(d) 
        baris.append([dn, pn, cv2.PSNR(bersih, out), ukur_waktu(f, d), 
                      kontras_bercak(out, md, mb)]) 
tabel = pd.DataFrame(baris, columns=["derau", "penapis", "PSNR_dB", 
                                     "waktu_ms", "KB"]).round(2) 
print(tabel) 
tabel.to_csv("../hasil/b4_tabel_penapis.csv", index=False) 

       derau   penapis  PSNR_dB  waktu_ms    KB
0   gauss_10     tanpa    28.11      0.00  3.39
1   gauss_10  rerata_3    36.93      0.29  5.67
2   gauss_10  rerata_5    37.90      0.45  5.89
3   gauss_10   gauss_5    37.79      0.17  5.84
4   gauss_10  median_3    35.50      0.10  5.52
5   gauss_10  median_5    38.42      0.66  6.32
6   gauss_10  median_9    38.63     10.11  6.91
7   gauss_20     tanpa    22.11      0.00  1.91
8   gauss_20  rerata_3    31.43      0.38  4.39
9   gauss_20  rerata_5    34.61      0.38  5.25
10  gauss_20   gauss_5    32.58      0.13  4.71
11  gauss_20  median_3    29.74      0.07  3.91
12  gauss_20  median_5    33.48      0.59  5.10
13  gauss_20  median_9    35.66     12.89  5.74
14  gauss_40     tanpa    16.38      0.00  0.96
15  gauss_40  rerata_3    25.84      0.33  2.70
16  gauss_40  rerata_5    29.82      0.58  3.84
17  gauss_40   gauss_5    27.06      0.22  3.03
18  gauss_40  median_3    23.83      0.06  2.27
19  gauss_40  median_5    27.85      0.5

# Kode 4.3 Pengaruh ukuran kernel median pada bercak kecil 

In [8]:
def selisih_rerata(kanal, mask_daun, mask_bercak): 
    """dI = |rerata bercak - rerata daun sehat|, dalam satuan intensitas.""" 
    k = kanal.astype(float) 
    sehat = (mask_daun > 0) & (mask_bercak == 0) 
    return abs(k[mask_bercak > 0].mean() - k[sehat].mean()) 
  
label = pd.read_csv("../data/sintetis/label.csv") 
baris = [] 
for nm in label[label.kelas == "ringan"].nama_file: 
    g = cv2.cvtColor(baca(f"../data/sintetis/bersih/{nm}"), cv2.COLOR_BGR2GRAY) 
    md2 = cv2.imread(f"../data/sintetis/mask_daun/{nm}", 0) 
    mb2 = cv2.imread(f"../data/sintetis/mask_bercak/{nm}", 0) 
    for k in [1, 3, 5, 9, 15]:          # k = 1 berarti tanpa penapis 
        out = g if k == 1 else cv2.medianBlur(g, k) 
        baris.append([nm, k, selisih_rerata(out, md2, mb2), 
                      kontras_bercak(out, md2, mb2)]) 
hasil_median = pd.DataFrame(baris, columns=["nama", "kernel", "dI", "KB"]) 
print(hasil_median.groupby("kernel")[["dI", "KB"]].mean().round(2)) 

           dI    KB
kernel             
1       30.62  4.81
3       31.04  4.95
5       30.18  4.88
9       24.00  4.45
15       8.36  2.27


# Kode 4.4 Pengaruh urutan penajaman 

In [9]:
def unsharp(x, k=1.0): 
    blur = cv2.GaussianBlur(x, (5, 5), 1.0) 
    return cv2.addWeighted(x, 1 + k, blur, -k, 0) 
  
noisy = derau["gauss_20"] 
latar = md == 0                         # area latar yang seharusnya rata 
for nm, redam in {"Gaussian": lambda x: cv2.GaussianBlur(x, (5, 5), 1.0), 
                  "Median": lambda x: cv2.medianBlur(x, 5)}.items(): 
    A = redam(unsharp(noisy))           # tajamkan dulu, lalu redam 
    B = unsharp(redam(noisy))           # redam dulu, lalu tajamkan 
    print(f"{nm}: tajam->redam PSNR={cv2.PSNR(bersih, A):.2f} " 
          f"sigma_latar={A[latar].std():.1f} | redam->tajam " 
          f"PSNR={cv2.PSNR(bersih, B):.2f} sigma_latar={B[latar].std():.1f}") 

Gaussian: tajam->redam PSNR=30.35 sigma_latar=7.9 | redam->tajam PSNR=30.20 sigma_latar=8.2
Median: tajam->redam PSNR=30.16 sigma_latar=8.1 | redam->tajam PSNR=31.13 sigma_latar=7.2
